In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
# Imports the create_engine function from the SQLAlchemy library so you can connect your Python program to a database.abs

import logging                 #standrad lib for storing the logs
import time                      #for keeping record of time

logging.basicConfig(
    filename="logs/ingestion_db.log",                      #where these logs are stored
    level=logging.DEBUG,                    #type of log(debug, error, info, )
    format="%(asctime)s - %(levelname)s - %(message)s",         #log store format 
    filemode="a"            #append after the existing log
)

engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, engine):
    ''' This function will ingest the dataframe into database table'''
    df.to_sql(table_name, con=engine, if_exists = 'replace', index = False)
    # ingest_db() saves a Pandas DataFrame into a SQL database table using a SQLAlchemy database connection.

def load_raw_data():
    ''' This function will load the CSVs as dataframe and ingest into db'''
    start = time.time()
    for file in os.listdir('data'):   #it brings all the file from data folder
        if '.csv' in file:            #it filters only the file which contains .csv
            print(file)
            df = pd.read_csv('data/'+file)             #it creates data frame from .csv file
            logging.info(f'Ingesting {file} in db')                   #gives shape of file (row, col)
            ingest_db(df, file[:-4], engine)   #file[:-4] removes .csv from last and give database name same as file name       
    end = time.time()
    total_time = (end-start)/60
    logging.info('Ingestion Complete')
    logging.info(f'Total time taken: {total_time} minutes')

if __name__ == '__main__':
    load_raw_data()

begin_inventory.csv
end_inventory.csv
purchases.csv


begin_inventory.csv
(206529, 9)
end_inventory.csv
(224489, 9)
purchases.csv
(2372474, 16)
purchase_prices.csv
(12261, 9)
sales.csv
(12825363, 14)
vendor_invoice.csv
(5543, 10)


In [7]:
# to do repetative tasks, we do scripting in python
import sqlite3
import pandas as pd
import logging
from ingestion_db import ingest_db

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a",  
    force = True
)

def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''
    vendor_sales_summary = pd.read_sql_query("""WITH FreightSummary AS (
    select 
        VendorNumber, 
        SUM(Freight) as FreightCost
    From vendor_invoice
    Group BY VendorNumber
    ),

    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Volume,
            pp.Price as ActualPrice,
            SUM(p.Quantity) as TotalPurchaseQuantity,
            SUM(p.Dollars) as TotalPurchaseDollars
    FROM purchases p
        JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.vendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
    ),

    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesDollars) as TotalSalesDollars,
            SUM(SalesPrice) as TotalSalesPrice,
            SUM(SalesQuantity) as TotalSalesQuantity,
            SUM(ExciseTax) as TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    )

    SELECT 
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps 
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC""", conn)
    return vendor_sales_summary

def clean_data(df):
    '''this function will clean the data'''
    # changing data type to float
    df['Volume'] = df['Volume'].astype('float')

    # filling missing value with 0
    df.fillna(0, inplace = True)

    # removing spaces from categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    # creating new columns for better analysis
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (df['GrossProfit']/df['TotalSalesDollars'])*100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

    return df

    

if __name__ == '__main__':
    #creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())
    
    logging.info('Ingesting data....')
    ingest_db(clean_df, 'vendor_sales_summary', conn)
    logging.info('Completed')

